In [8]:
!pip install grad-cam

In [9]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import cv2
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms, models
from torch.optim.lr_scheduler import ReduceLROnPlateau, StepLR

# TensorBoard
from torch.utils.tensorboard import SummaryWriter

# Grad-CAM
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

# Sklearn
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

# Settings
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using: {device}')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

AttributeError: partially initialized module 'torch' has no attribute 'fx' (most likely due to a circular import)

2. Exploratory Data Analysis (EDA)

In [ ]:

TRAIN_DIR = '/kaggle/input/datasets/shashwatwork/knee-osteoarthritis-dataset-with-severity/train'
TEST_DIR = '/kaggle/input/datasets/shashwatwork/knee-osteoarthritis-dataset-with-severity/test'
VAL_DIR = '/kaggle/input/datasets/shashwatwork/knee-osteoarthritis-dataset-with-severity/val'

def analyze_dataset(data_dir):
    class_counts = {}
    total_images = 0
    image_sizes = []
    
    for class_name in os.listdir(data_dir):
        class_path = os.path.join(data_dir, class_name)
        if os.path.isdir(class_path):
            images = os.listdir(class_path)
            class_counts[class_name] = len(images)
            total_images += len(images)
            
            for img_name in images[:10]:
                img_path = os.path.join(class_path, img_name)
                try:
                    img = Image.open(img_path)
                    image_sizes.append(img.size)
                except:
                    pass
    
    return class_counts, total_images, image_sizes

# train, val, test
train_counts, train_total, train_sizes = analyze_dataset(TRAIN_DIR)
val_counts, val_total, val_sizes = analyze_dataset(VAL_DIR)
test_counts, test_total, test_sizes = analyze_dataset(TEST_DIR)

print("Datasets analysis")
print(f"\nTrain: {train_total} images")
print(f"Validation: {val_total} images")
print(f"Test: {test_total} images")
print(f"Total: {train_total + val_total + test_total} images\n")

In [ ]:
# Visualize class distribution
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

datasets_info = [
    ('Train', train_counts),
    ('Validation', val_counts),
    ('Test', test_counts)
]

for ax, (title, counts) in zip(axes, datasets_info):
    classes = sorted(counts.keys())
    values = [counts[c] for c in classes]
    
    bars = ax.bar(classes, values, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd'])
    ax.set_title(f'{title} Dataset Distribution', fontsize=14, fontweight='bold')
    ax.set_xlabel('KL Grade (Osteoarthritis Class)', fontsize=12)
    ax.set_ylabel('Number of Images', fontsize=12)
    ax.grid(axis='y', alpha=0.3)
    
    # Add value labels on top of bars
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{int(height)}',
                ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

# Check for class imbalance
print("\nClass distribution in the training set:")
for class_name, count in sorted(train_counts.items()):
    percentage = (count / train_total) * 100
    print(f"Class {class_name}: {count} ({percentage:.1f}%)")

# Determine if balancing is necessary
max_count = max(train_counts.values())
min_count = min(train_counts.values())
imbalance_ratio = max_count / min_count
print(f"\nImbalance Ratio: {imbalance_ratio:.2f}")

if imbalance_ratio > 2:
    print("Significant class imbalance detected! Using WeightedRandomSampler or data augmentation is recommended.")
else:
    print("The dataset is relatively balanced.")

In [ ]:
# Image size analysis
all_sizes = train_sizes + val_sizes + test_sizes
widths = [s[0] for s in all_sizes]
heights = [s[1] for s in all_sizes]

print("IMAGE SIZE ANALYSIS")
print(f"Mean width: {np.mean(widths):.0f} px")
print(f"Mean height: {np.mean(heights):.0f} px")
print(f"Min. size: {min(widths)}x{min(heights)} px")
print(f"Max. size: {max(widths)}x{max(heights)} px")
print(f"Most common size: {Counter(all_sizes).most_common(1)[0][0]}")

In [ ]:
# Image sizes

def get_image_sizes(data_dir):
    sizes = []

    for class_name in sorted(os.listdir(data_dir)):
        class_path = os.path.join(data_dir, class_name)

        if not os.path.isdir(class_path):
            continue

        for img_name in os.listdir(class_path):
            img_path = os.path.join(class_path, img_name)

            try:
                with Image.open(img_path) as img:
                    sizes.append(img.size)
            except Exception as e:
                print(f"Error reading {img_path}: {e}")

    return sizes


train_sizes = get_image_sizes(TRAIN_DIR)

size_counts = Counter(train_sizes)

print("Unique image sizes:")
for size, count in size_counts.items():
    print(size, count)

In [ ]:
# Visualization of samples from each class
fig, axes = plt.subplots(5, 5, figsize=(15, 15))
fig.suptitle('Sample Images from Each Class (KL Grade 0-4)', fontsize=16, fontweight='bold')

for class_idx in range(5):
    class_dir = os.path.join(TRAIN_DIR, str(class_idx))
    images = os.listdir(class_dir)[:5]
    
    for img_idx, img_name in enumerate(images):
        img_path = os.path.join(class_dir, img_name)
        img = Image.open(img_path)
        
        ax = axes[class_idx, img_idx]
        ax.imshow(img, cmap='gray')
        ax.axis('on')
        
        # Adding class labels for the first image in each row
        if img_idx == 0:
            ax.set_ylabel(f'KL Grade {class_idx}', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('sample_images.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
def analyze_brightness_contrast(data_dir):
    records = []

    for class_name in sorted(os.listdir(data_dir)):
        class_path = os.path.join(data_dir, class_name)

        if not os.path.isdir(class_path):
            continue

        for img_name in os.listdir(class_path):
            img_path = os.path.join(class_path, img_name)

            try:
                img = Image.open(img_path).convert("L")
                arr = np.array(img, dtype=np.float32)

                records.append({
                    "class": int(class_name),
                    "image": img_name,
                    "mean_intensity": arr.mean(),
                    "std_intensity": arr.std()
                })

            except Exception as e:
                print(f"Error reading {img_path}: {e}")

    return pd.DataFrame(records)


stats_df = analyze_brightness_contrast(TRAIN_DIR)

print(stats_df.groupby("class")[["mean_intensity", "std_intensity"]].agg(
    ["mean", "std", "min", "max"]
))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

stats_df.boxplot(column="mean_intensity", by="class", ax=axes[0])
axes[0].set_title("Brightness by KL Grade")
axes[0].set_xlabel("KL Grade")
axes[0].set_ylabel("Mean pixel intensity")

stats_df.boxplot(column="std_intensity", by="class", ax=axes[1])
axes[1].set_title("Contrast by KL Grade")
axes[1].set_xlabel("KL Grade")
axes[1].set_ylabel("Pixel intensity std")

plt.suptitle("")
plt.tight_layout()
plt.show()

In [ ]:
balanced_stats = (
    stats_df
    .groupby("class", group_keys=False)
    .sample(n=150, random_state=42)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

balanced_stats.boxplot(
    column="mean_intensity",
    by="class",
    ax=axes[0]
)
axes[0].set_title("Brightness by KL Grade")
axes[0].set_xlabel("KL Grade")
axes[0].set_ylabel("Mean pixel intensity")

balanced_stats.boxplot(
    column="std_intensity",
    by="class",
    ax=axes[1]
)
axes[1].set_title("Contrast by KL Grade")
axes[1].set_xlabel("KL Grade")
axes[1].set_ylabel("Pixel intensity std")

plt.suptitle("")
plt.tight_layout()
plt.show()

### Searching for duplicates

In [ ]:
import hashlib

def get_image_hashes(data_dir, split_name):
    records = []

    for class_name in sorted(os.listdir(data_dir)):
        class_path = os.path.join(data_dir, class_name)

        if not os.path.isdir(class_path):
            continue

        for img_name in os.listdir(class_path):
            img_path = os.path.join(class_path, img_name)

            try:
                with open(img_path, "rb") as f:
                    file_hash = hashlib.md5(f.read()).hexdigest()

                records.append({
                    "split": split_name,
                    "class": class_name,
                    "image": img_name,
                    "path": img_path,
                    "hash": file_hash
                })

            except Exception as e:
                print(e)

    return records


all_hashes = (
    get_image_hashes(TRAIN_DIR, "train")
    + get_image_hashes(VAL_DIR, "val")
    + get_image_hashes(TEST_DIR, "test")
)

hash_df = pd.DataFrame(all_hashes)

duplicates = hash_df[hash_df.duplicated("hash", keep=False)].sort_values("hash")

print(f"Duplicate image entries: {len(duplicates)}")
display(duplicates)

### EDA Conclusions

Based on the exploratory dataset analysis:

1. **Dataset size:** The number of available images is sufficient for transfer learning and fine-tuning experiments.
2. **Class imbalance:** The dataset is strongly imbalanced, especially for KL Grade 4, so class weighting, sampling, or augmentation should be considered during training.
3. **Image characteristics:** Images have a consistent size of 224 × 224 pixels. Brightness and contrast distributions largely overlap across KL grades, although several outliers are present.
4. **Data integrity:** No exact duplicates were detected across the train, validation, and test splits.

## Data preparation and augumentation

In [ ]:
import numpy as np
import torch

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, WeightedRandomSampler


# Parameters
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 5


# Transformations

# Training set transformations with mild augmentation
train_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),

    transforms.Resize((IMG_SIZE, IMG_SIZE)),

    transforms.RandomRotation(7),

    transforms.RandomHorizontalFlip(p=0.5),

    transforms.RandomAffine(
        degrees=0,
        translate=(0.05, 0.05)
    ),

    transforms.ColorJitter(
        brightness=0.1,
        contrast=0.1
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# Validation/test: no augmentation
val_test_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),

    transforms.Resize((IMG_SIZE, IMG_SIZE)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# Datasets

train_dataset = datasets.ImageFolder(
    root=TRAIN_DIR,
    transform=train_transforms
)

val_dataset = datasets.ImageFolder(
    root=VAL_DIR,
    transform=val_test_transforms
)

test_dataset = datasets.ImageFolder(
    root=TEST_DIR,
    transform=val_test_transforms
)


print(f"Training set: {len(train_dataset)} images")
print(f"Validation set: {len(val_dataset)} images")
print(f"Test set: {len(test_dataset)} images")
print(f"Classes: {train_dataset.class_to_idx}")


# WeightedRandomSampler

targets = np.array(train_dataset.targets)

class_counts = np.bincount(targets)

# Inverse frequency
class_weights = 1.0 / class_counts

# Weight for every training image
sample_weights = class_weights[targets]

sampler = WeightedRandomSampler(
    weights=torch.as_tensor(
        sample_weights,
        dtype=torch.double
    ),
    num_samples=len(train_dataset),
    replacement=True
)


print("\nOriginal class distribution:")
for class_idx, count in enumerate(class_counts):
    print(f"KL {class_idx}: {count}")

print("\nClass sampling weights:")
for class_idx, weight in enumerate(class_weights):
    print(f"KL {class_idx}: {weight:.6f}")


# DataLoaders

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# ResNet with transfer learning

In [ ]:
def create_resnet50_model(num_classes=5, freeze_backbone=True):
    """
    Resnet 50 creation with pretrained weights
    """

    model = models.resnet50(pretrained=True)

    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False
    num_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(num_features, 512),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(512, num_classes)
    )
    
    return model

In [ ]:
resnet_model = create_resnet50_model(num_classes=NUM_CLASSES, freeze_backbone=True)
resnet_model = resnet_model.to(device)

# Display model information
total_params = sum(p.numel() for p in resnet_model.parameters())
trainable_params = sum(p.numel() for p in resnet_model.parameters() if p.requires_grad)

print(f"\nResulting ResNet50 Model:")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters: {total_params - trainable_params:,}")

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
import torch.nn.functional as F

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)

        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
    # FIX: Use division instead of addition for average loss
    epoch_loss = running_loss / len(loader)
    epoch_acc = 100. * correct / total
    return epoch_loss, epoch_acc

def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    all_preds = []
    all_labels = []
    all_probs = [] # Necessary for AUC calculation

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            
            # Get probabilities using Softmax
            probs = F.softmax(outputs, dim=1)
            _, predicted = outputs.max(1)
            
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    
    # Calculate main metrics
    epoch_loss = running_loss / len(loader)
    epoch_acc = 100. * correct / total
    
    # Calculate advanced metrics
    # Using 'weighted' average to account for class imbalance
    precision = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
    recall = recall_score(all_labels, all_preds, average='weighted')
    f1 = f1_score(all_labels, all_preds, average='weighted')
    
    # Multi-class AUC (One-vs-Rest strategy)
    try:
        auc = roc_auc_score(all_labels, all_probs, multi_class='ovr', average='weighted')
    except ValueError:
        auc = 0.0 # Handle cases where not all classes are present in a batch/subset

    metrics = {
        'loss': epoch_loss,
        'accuracy': epoch_acc,
        'precision': precision * 100,
        'recall': recall * 100,
        'f1': f1 * 100,
        'auc': auc
    }
    
    return metrics, all_preds, all_labels

## ResNet50 Phase 1: head only

In [ ]:
# Phase 1 Settings
PHASE1_EPOCHS = 15
PHASE1_LR = 0.001

criterion = nn.CrossEntropyLoss()
optimizer_phase1 = optim.Adam(resnet_model.fc.parameters(), lr=PHASE1_LR)
scheduler_phase1 = ReduceLROnPlateau(optimizer_phase1, mode='min', factor=0.5, patience=3)

# TensorBoard setup
writer_resnet_phase1 = SummaryWriter('runs/resnet50_phase1')

# Phase 1 Training
print("\n" + "="*60)
print("PHASE 1: Training the classification head only")
print("="*60)

best_val_f1 = 0.0
history_phase1 = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'val_f1': []}

for epoch in range(PHASE1_EPOCHS):
    # Training
    train_loss, train_acc = train_epoch(resnet_model, train_loader, criterion, optimizer_phase1, device)
    
    # Validation - unpacking the new dictionary format
    val_metrics, _, _ = validate(resnet_model, val_loader, criterion, device)
    
    # Save history
    history_phase1['train_loss'].append(train_loss)
    history_phase1['train_acc'].append(train_acc)
    history_phase1['val_loss'].append(val_metrics['loss'])
    history_phase1['val_acc'].append(val_metrics['accuracy'])
    history_phase1['val_f1'].append(val_metrics['f1'])
    
    # Log all metrics to TensorBoard
    writer_resnet_phase1.add_scalar('Loss/train', train_loss, epoch)
    writer_resnet_phase1.add_scalar('Loss/val', val_metrics['loss'], epoch)
    writer_resnet_phase1.add_scalar('Accuracy/train', train_acc, epoch)
    writer_resnet_phase1.add_scalar('Accuracy/val', val_metrics['accuracy'], epoch)
    writer_resnet_phase1.add_scalar('F1-Score/val', val_metrics['f1'], epoch)
    writer_resnet_phase1.add_scalar('Precision/val', val_metrics['precision'], epoch)
    writer_resnet_phase1.add_scalar('Recall/val', val_metrics['recall'], epoch)
    writer_resnet_phase1.add_scalar('AUC/val', val_metrics['auc'], epoch)
    
    # Update scheduler based on validation loss
    scheduler_phase1.step(val_metrics['loss'])
    
    # Save the best model based on F1-Score (often better for imbalanced data)
    if val_metrics['f1'] > best_val_f1:
        best_val_f1 = val_metrics['f1']
        torch.save(resnet_model.state_dict(), 'resnet50_phase1_best.pth')
        print(f"✓ Best model saved (Epoch {epoch+1}) with F1: {best_val_f1:.2f}%")
    
    print(f"Epoch {epoch+1}/{PHASE1_EPOCHS}:")
    print(f"  Train Loss: {train_loss:.4f} | Acc: {train_acc:.2f}%")
    print(f"  Val Loss: {val_metrics['loss']:.4f} | Acc: {val_metrics['accuracy']:.2f}% | F1: {val_metrics['f1']:.2f}%")
    print(f"  AUC: {val_metrics['auc']:.4f}")
    print("-" * 60)

writer_resnet_phase1.close()
print(f"\nBest Phase 1 F1-Score: {best_val_f1:.2f}%")

## ResNet50 Phase 2: fine-tuning

Strategy,What happens?,Pros,Cons
"Option A: ""Default"" (Fully Trainable)",You load the model and set requires_grad=True for everything immediately.,Faster to setup; might reach a solution quicker if the data is large.,"Risk of ""Catastrophic Forgetting""—the model might destroy useful ImageNet features before adapting to X-rays."
Option B: Your 2-Phase Strategy,"Phase 1 trains only the ""head."" Phase 2 fine-tunes the ""body"" (Layers 3 & 4).",Much more stable. Preserves general features (edges/textures) while specializing high-level features (bone shapes).,Requires more code and two separate training runs.

In [ ]:
# Unfreezing the last few layers
def unfreeze_last_layers(model, num_layers=2):
    """
    Unfreezes the last num_layers blocks of the ResNet model.
    """
    # ResNet has 4 main layer blocks: layer1, layer2, layer3, layer4
    layers_to_unfreeze = []
    if num_layers >= 1:
        layers_to_unfreeze.append(model.layer4)
    if num_layers >= 2:
        layers_to_unfreeze.append(model.layer3)
    if num_layers >= 3:
        layers_to_unfreeze.append(model.layer2)
    if num_layers >= 4:
        layers_to_unfreeze.append(model.layer1)

    for layer in layers_to_unfreeze:
        for param in layer.parameters():
            param.requires_grad = True

    return model

# Load the best model from Phase 1
resnet_model.load_state_dict(torch.load('resnet50_phase1_best.pth'))

# Unfreeze the last 2 blocks (layer3 and layer4)
resnet_model = unfreeze_last_layers(resnet_model, num_layers=2)

# Verification
total_params = sum(p.numel() for p in resnet_model.parameters())
trainable_params = sum(p.numel() for p in resnet_model.parameters() if p.requires_grad)

print(f"\nAfter unfreezing:")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters: {total_params - trainable_params:,}")

In [ ]:
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0):
        """
        Stop training when validation loss stops improving.
        """
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.counter = 0

In [ ]:
# Phase 2 Settings
PHASE2_EPOCHS = 15
PHASE2_LR = 0.0001  # Lower learning rate for fine-tuning!

# Optimizer only for parameters that require gradients
optimizer_phase2 = optim.Adam(filter(lambda p: p.requires_grad, resnet_model.parameters()), lr=PHASE2_LR)
scheduler_phase2 = ReduceLROnPlateau(optimizer_phase2, mode='min', factor=0.5, patience=3)

# Early stopping setup
early_stopping = EarlyStopping(patience=5)

# TensorBoard setup
writer_resnet_phase2 = SummaryWriter('runs/resnet50_phase2')

# Phase 2 Training
print("\n" + "="*60)
print("PHASE 2: Fine-tuning (unfrozen layer3 and layer4)")
print("="*60)

best_val_f1_phase2 = 0.0
history_phase2 = {
    'train_loss': [], 'train_acc': [], 
    'val_loss': [], 'val_acc': [], 'val_f1': []
}

for epoch in range(PHASE2_EPOCHS):
    # Training
    train_loss, train_acc = train_epoch(resnet_model, train_loader, criterion, optimizer_phase2, device)
    
    # Validation - unpacking the metrics dictionary
    val_metrics, _, _ = validate(resnet_model, val_loader, criterion, device)

    # Save history
    history_phase2['train_loss'].append(train_loss)
    history_phase2['train_acc'].append(train_acc)
    history_phase2['val_loss'].append(val_metrics['loss'])
    history_phase2['val_acc'].append(val_metrics['accuracy'])
    history_phase2['val_f1'].append(val_metrics['f1'])

    # Log all metrics to TensorBoard
    writer_resnet_phase2.add_scalar('Loss/train', train_loss, epoch)
    writer_resnet_phase2.add_scalar('Loss/val', val_metrics['loss'], epoch)
    writer_resnet_phase2.add_scalar('Accuracy/train', train_acc, epoch)
    writer_resnet_phase2.add_scalar('Accuracy/val', val_metrics['accuracy'], epoch)
    writer_resnet_phase2.add_scalar('F1-Score/val', val_metrics['f1'], epoch)
    writer_resnet_phase2.add_scalar('Precision/val', val_metrics['precision'], epoch)
    writer_resnet_phase2.add_scalar('Recall/val', val_metrics['recall'], epoch)
    writer_resnet_phase2.add_scalar('AUC/val', val_metrics['auc'], epoch)

    # Update scheduler
    scheduler_phase2.step(val_metrics['loss'])

    # Early stopping check based on validation loss
    early_stopping(val_metrics['loss'])
    if early_stopping.early_stop:
        print("\n⚠️ Early stopping triggered")
        break

    # Save the best model based on F1-Score
    if val_metrics['f1'] > best_val_f1_phase2:
        best_val_f1_phase2 = val_metrics['f1']
        torch.save(resnet_model.state_dict(), 'resnet50_phase2_best.pth')
        print(f"✓ Best model saved (Epoch {epoch+1}) with F1: {best_val_f1_phase2:.2f}%")

    print(f"Epoch {epoch+1}/{PHASE2_EPOCHS}:")
    print(f"  Train Loss: {train_loss:.4f} | Acc: {train_acc:.2f}%")
    print(f"  Val Loss: {val_metrics['loss']:.4f} | Acc: {val_metrics['accuracy']:.2f}% | F1: {val_metrics['f1']:.2f}%")
    print(f"  AUC: {val_metrics['auc']:.4f}")
    print("-" * 60)

writer_resnet_phase2.close()
print(f"\nBest Phase 2 F1-Score: {best_val_f1_phase2:.2f}%")

# ViT with transfer learning

In [ ]:
def create_vit_model(num_classes=5, freeze_backbone=True):
    """
    Creates a Vision Transformer (ViT) with pre-trained weights.
    """
    # Using vit_b_16 (Base model with 16x16 patch size)
    model = models.vit_b_16(weights='DEFAULT')

    # Freeze all layers if requested
    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False

    # Replace the classification head
    # ViT in torchvision uses model.heads.head for the final layer
    num_features = model.heads.head.in_features
    model.heads.head = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(num_features, 512),
        nn.GELU(),  # GELU is standard for Transformers
        nn.Dropout(0.3),
        nn.Linear(512, num_classes)
    )

    return model

# Create the model
vit_model = create_vit_model(num_classes=NUM_CLASSES, freeze_backbone=True)
vit_model = vit_model.to(device)

# Display model information
total_params = sum(p.numel() for p in vit_model.parameters())
trainable_params = sum(p.numel() for p in vit_model.parameters() if p.requires_grad)

print(f"\nResulting ViT Model:")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters: {total_params - trainable_params:,}")

## ViT Phase 1: head only

In [ ]:
# ViT Phase 1 Settings
VIT_PHASE1_EPOCHS = 10
VIT_PHASE1_LR = 0.001

# Only optimize the classification head parameters
optimizer_vit_phase1 = optim.Adam(vit_model.heads.head.parameters(), lr=VIT_PHASE1_LR)
scheduler_vit_phase1 = ReduceLROnPlateau(optimizer_vit_phase1, mode='min', factor=0.5, patience=3)

# TensorBoard setup
writer_vit_phase1 = SummaryWriter('runs/vit_phase1')

# Training
print("\n" + "="*60)
print("ViT PHASE 1: Training the classification head only")
print("="*60)

best_val_f1_vit_phase1 = 0.0
history_vit_phase1 = {
    'train_loss': [], 'train_acc': [], 
    'val_loss': [], 'val_acc': [], 'val_f1': []
}

for epoch in range(VIT_PHASE1_EPOCHS):
    # Training
    train_loss, train_acc = train_epoch(vit_model, train_loader, criterion, optimizer_vit_phase1, device)
    
    # Validation - unpacking the new dictionary format
    val_metrics, _, _ = validate(vit_model, val_loader, criterion, device)

    # Save history
    history_vit_phase1['train_loss'].append(train_loss)
    history_vit_phase1['train_acc'].append(train_acc)
    history_vit_phase1['val_loss'].append(val_metrics['loss'])
    history_vit_phase1['val_acc'].append(val_metrics['accuracy'])
    history_vit_phase1['val_f1'].append(val_metrics['f1'])

    # Log all metrics to TensorBoard
    writer_vit_phase1.add_scalar('Loss/train', train_loss, epoch)
    writer_vit_phase1.add_scalar('Loss/val', val_metrics['loss'], epoch)
    writer_vit_phase1.add_scalar('Accuracy/train', train_acc, epoch)
    writer_vit_phase1.add_scalar('Accuracy/val', val_metrics['accuracy'], epoch)
    writer_vit_phase1.add_scalar('F1-Score/val', val_metrics['f1'], epoch)
    writer_vit_phase1.add_scalar('Precision/val', val_metrics['precision'], epoch)
    writer_vit_phase1.add_scalar('Recall/val', val_metrics['recall'], epoch)
    writer_vit_phase1.add_scalar('AUC/val', val_metrics['auc'], epoch)

    # Update scheduler
    scheduler_vit_phase1.step(val_metrics['loss'])

    # Save the best model based on F1-Score
    if val_metrics['f1'] > best_val_f1_vit_phase1:
        best_val_f1_vit_phase1 = val_metrics['f1']
        torch.save(vit_model.state_dict(), 'vit_phase1_best.pth')
        print(f"✓ Best model saved (Epoch {epoch+1}) with F1: {best_val_f1_vit_phase1:.2f}%")

    print(f"Epoch {epoch+1}/{VIT_PHASE1_EPOCHS}:")
    print(f"  Train Loss: {train_loss:.4f} | Acc: {train_acc:.2f}%")
    print(f"  Val Loss: {val_metrics['loss']:.4f} | Acc: {val_metrics['accuracy']:.2f}% | F1: {val_metrics['f1']:.2f}%")
    print(f"  AUC: {val_metrics['auc']:.4f}")
    print("-" * 60)

writer_vit_phase1.close()
print(f"\nBest ViT Phase 1 F1-Score: {best_val_f1_vit_phase1:.2f}%")

## ViT Phase 2: fine-tuning

In [ ]:
# Load the best model from Phase 1
vit_model.load_state_dict(torch.load('vit_phase1_best.pth'))

# Unfreeze the last few encoder blocks
# In vit_b_16, there are 12 encoder blocks (layers) in total.
num_blocks_to_unfreeze = 4
total_blocks = len(vit_model.encoder.layers)

for i in range(total_blocks - num_blocks_to_unfreeze, total_blocks):
    for param in vit_model.encoder.layers[i].parameters():
        param.requires_grad = True

# Verification
total_params = sum(p.numel() for p in vit_model.parameters())
trainable_params = sum(p.numel() for p in vit_model.parameters() if p.requires_grad)

print(f"\nAfter unfreezing ViT:")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters: {total_params - trainable_params:,}")

In [ ]:
# ViT Phase 2 Settings
VIT_PHASE2_EPOCHS = 15
VIT_PHASE2_LR = 0.0001 

# Оптимізатор тільки для параметрів, що потребують градієнтів (4 блоки + head)
optimizer_vit_phase2 = optim.Adam(filter(lambda p: p.requires_grad, vit_model.parameters()), lr=VIT_PHASE2_LR)
scheduler_vit_phase2 = ReduceLROnPlateau(optimizer_vit_phase2, mode='min', factor=0.5, patience=3)
early_stopping_vit = EarlyStopping(patience=5)

# TensorBoard setup
writer_vit_phase2 = SummaryWriter('runs/vit_phase2')

# Training loop
print("\n" + "="*60)
print("ViT PHASE 2: Fine-tuning (unfrozen last 4 encoder blocks)")
print("="*60)

best_val_f1_vit_phase2 = 0.0
history_vit_phase2 = {
    'train_loss': [], 'train_acc': [], 
    'val_loss': [], 'val_acc': [], 'val_f1': []
}

for epoch in range(VIT_PHASE2_EPOCHS):
    # Training
    train_loss, train_acc = train_epoch(vit_model, train_loader, criterion, optimizer_vit_phase2, device)
    
    # Validation - розпакування словника метрик
    val_metrics, _, _ = validate(vit_model, val_loader, criterion, device)

    # Save history
    history_vit_phase2['train_loss'].append(train_loss)
    history_vit_phase2['train_acc'].append(train_acc)
    history_vit_phase2['val_loss'].append(val_metrics['loss'])
    history_vit_phase2['val_acc'].append(val_metrics['accuracy'])
    history_vit_phase2['val_f1'].append(val_metrics['f1'])

    # Log metrics to TensorBoard
    writer_vit_phase2.add_scalar('Loss/train', train_loss, epoch)
    writer_vit_phase2.add_scalar('Loss/val', val_metrics['loss'], epoch)
    writer_vit_phase2.add_scalar('Accuracy/train', train_acc, epoch)
    writer_vit_phase2.add_scalar('Accuracy/val', val_metrics['accuracy'], epoch)
    writer_vit_phase2.add_scalar('F1-Score/val', val_metrics['f1'], epoch)
    writer_vit_phase2.add_scalar('Precision/val', val_metrics['precision'], epoch)
    writer_vit_phase2.add_scalar('Recall/val', val_metrics['recall'], epoch)
    writer_vit_phase2.add_scalar('AUC/val', val_metrics['auc'], epoch)

    # Update scheduler
    scheduler_vit_phase2.step(val_metrics['loss'])

    # Early stopping check
    early_stopping_vit(val_metrics['loss'])
    if early_stopping_vit.early_stop:
        print("\n⚠️ Early stopping triggered")
        break

    # Save best model based on F1-Score
    if val_metrics['f1'] > best_val_f1_vit_phase2:
        best_val_f1_vit_phase2 = val_metrics['f1']
        torch.save(vit_model.state_dict(), 'vit_phase2_best.pth')
        print(f"✓ Best model saved (Epoch {epoch+1}) with F1: {best_val_f1_vit_phase2:.2f}%")

    print(f"Epoch {epoch+1}/{VIT_PHASE2_EPOCHS}:")
    print(f"  Train Loss: {train_loss:.4f} | Acc: {train_acc:.2f}%")
    print(f"  Val Loss: {val_metrics['loss']:.4f} | Acc: {val_metrics['accuracy']:.2f}% | F1: {val_metrics['f1']:.2f}%")
    print(f"  AUC: {val_metrics['auc']:.4f}")
    print("-" * 60)

writer_vit_phase2.close()
print(f"\nBest ViT Phase 2 F1-Score: {best_val_f1_vit_phase2:.2f}%")

# Models comparison

In [ ]:
# Comparison of training curves (Loss vs F1-Score)
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# --- ResNet50 ---
# ResNet Loss
axes[0, 0].plot(history_phase1['train_loss'] + history_phase2['train_loss'], label='Train Loss', marker='o')
axes[0, 0].plot(history_phase1['val_loss'] + history_phase2['val_loss'], label='Val Loss', marker='s')
axes[0, 0].axvline(x=len(history_phase1['train_loss']), color='r', linestyle='--', label='Phase 1→2')
axes[0, 0].set_title('ResNet50: Cross Entropy Loss', fontsize=14, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# ResNet F1-Score
# Note: Comparing Train Acc vs Val F1 helps see if general accuracy translates to class balance
axes[0, 1].plot(history_phase1['train_acc'] + history_phase2['train_acc'], label='Train Acc', alpha=0.5, linestyle='--')
axes[0, 1].plot(history_phase1['val_f1'] + history_phase2['val_f1'], label='Val F1-Score', marker='s', color='darkblue')
axes[0, 1].axvline(x=len(history_phase1['train_acc']), color='r', linestyle='--', label='Phase 1→2')
axes[0, 1].set_title('ResNet50: F1-Score (Weighted)', fontsize=14, fontweight='bold')
axes[0, 1].set_ylabel('Score (%)')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# --- Vision Transformer ---
# ViT Loss
axes[1, 0].plot(history_vit_phase1['train_loss'] + history_vit_phase2['train_loss'], label='Train Loss', marker='o', color='green')
axes[1, 0].plot(history_vit_phase1['val_loss'] + history_vit_phase2['val_loss'], label='Val Loss', marker='s', color='orange')
axes[1, 0].axvline(x=len(history_vit_phase1['train_loss']), color='r', linestyle='--', label='Phase 1→2')
axes[1, 0].set_title('ViT: Cross Entropy Loss', fontsize=14, fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# ViT F1-Score
axes[1, 1].plot(history_vit_phase1['train_acc'] + history_vit_phase2['train_acc'], label='Train Acc', alpha=0.5, linestyle='--', color='green')
axes[1, 1].plot(history_vit_phase1['val_f1'] + history_vit_phase2['val_f1'], label='Val F1-Score', marker='s', color='darkgreen')
axes[1, 1].axvline(x=len(history_vit_phase1['train_acc']), color='r', linestyle='--', label='Phase 1→2')
axes[1, 1].set_title('ViT: F1-Score (Weighted)', fontsize=14, fontweight='bold')
axes[1, 1].set_ylabel('Score (%)')
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('f1_loss_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Load the best weights for both models
resnet_model.load_state_dict(torch.load('resnet50_phase2_best.pth'))
vit_model.load_state_dict(torch.load('vit_phase2_best.pth'))

def print_test_results(model_name, metrics):
    print(f"\n" + "="*40)
    print(f"RESULTS: {model_name}")
    print("="*40)
    print(f"Test Loss:    {metrics['loss']:.4f}")
    print(f"F1-Score:     {metrics['f1']:.2f}%")
    print(f"AUC-ROC:      {metrics['auc']:.4f}")
    print(f"Precision:    {metrics['precision']:.2f}%")
    print(f"Recall:       {metrics['recall']:.2f}%")
    print(f"Accuracy:     {metrics['accuracy']:.2f}%") # Still useful as a baseline
    print("-" * 40)

# Testing ResNet50
resnet_metrics, preds_resnet, labels_resnet = validate(resnet_model, test_loader, criterion, device)
print_test_results("ResNet50", resnet_metrics)

# Testing Vision Transformer
vit_metrics, preds_vit, labels_vit = validate(vit_model, test_loader, criterion, device)
print_test_results("Vision Transformer", vit_metrics)

# Final Comparison Table
import pandas as pd

comparison_df = pd.DataFrame({
    'Metric (%)': ['F1-Score', 'Precision', 'Recall', 'Accuracy'],
    'ResNet50': [resnet_metrics['f1'], resnet_metrics['precision'], resnet_metrics['recall'], resnet_metrics['accuracy']],
    'ViT': [vit_metrics['f1'], vit_metrics['precision'], vit_metrics['recall'], vit_metrics['accuracy']]
}).round(2)

print("\nFINAL HEAD-TO-HEAD COMPARISON")
print(comparison_df.to_string(index=False))

In [ ]:
# Classification Reports
class_names = ['KL-0', 'KL-1', 'KL-2', 'KL-3', 'KL-4']

print("\n" + "="*60)
print("CLASSIFICATION REPORT - ResNet50")
print("="*60)
print(classification_report(labels_resnet, preds_resnet, target_names=class_names))

print("\n" + "="*60)
print("CLASSIFICATION REPORT - Vision Transformer")
print("="*60)
print(classification_report(labels_vit, preds_vit, target_names=class_names))

In [ ]:
# Confusion Matrices
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# ResNet Confusion Matrix
cm_resnet = confusion_matrix(labels_resnet, preds_resnet)
sns.heatmap(cm_resnet, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=axes[0])
axes[0].set_title('ResNet50 Confusion Matrix', fontsize=14, fontweight='bold')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

# ViT Confusion Matrix
cm_vit = confusion_matrix(labels_vit, preds_vit)
sns.heatmap(cm_vit, annot=True, fmt='d', cmap='Greens',
            xticklabels=class_names, yticklabels=class_names, ax=axes[1])
axes[1].set_title('Vision Transformer Confusion Matrix', fontsize=14, fontweight='bold')
axes[1].set_ylabel('True Label')
axes[1].set_xlabel('Predicted Label')

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()

# Grand-Cam

In [ ]:
def vit_reshape_transform(tensor, height=14, width=14):
    # Видаляємо CLS-токен (перший токен у послідовності 197)
    result = tensor[:, 1:, :].reshape(tensor.size(0), 
                                      height, width, tensor.size(2))

    # Переставляємо розмірності, щоб отримати (Batch, Channel, Height, Width)
    result = result.permute(0, 3, 1, 2)
    return result

In [ ]:
# Function for Grad-CAM
# def apply_gradcam(model, input_tensor, target_layer, target_category=None):
#     """
#     Applies Grad-CAM to the model to generate an attention heatmap.
#     """
#     # Initialize the GradCAM object with the specified target layer
#     cam = GradCAM(model=model, target_layers=[target_layer])

#     # If a specific class is provided, target it; otherwise, use the highest-scoring class
#     targets = None
#     if target_category is not None:
#         targets = [ClassifierOutputTarget(target_category)]

#     # Generate the heatmap
#     grayscale_cam = cam(input_tensor=input_tensor, targets=targets)
    
#     # Extract the first image in the batch
#     grayscale_cam = grayscale_cam[0, :]

#     return grayscale_cam

def apply_gradcam(model, input_tensor, target_layer, reshape_transform=None, target_category=None):
    # Додаємо reshape_transform у конструктор GradCAM
    cam = GradCAM(model=model, 
                  target_layers=[target_layer], 
                  reshape_transform=reshape_transform)

    targets = None
    if target_category is not None:
        targets = [ClassifierOutputTarget(target_category)]

    grayscale_cam = cam(input_tensor=input_tensor, targets=targets)
    return grayscale_cam[0, :]
    
# Data preparation for visualization
def prepare_image_for_gradcam(image_path):
    """
    Loads and prepares an image for Grad-CAM visualization.
    """
    # Open image and normalize pixel values for the original image display
    img = Image.open(image_path).convert('RGB')
    img_array = np.array(img) / 255.0

    # Define the same transformations used during validation
    transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])

    # Create a batch of 1 for the model input
    input_tensor = transform(img).unsqueeze(0)
    return input_tensor, img_array

In [ ]:
# Grad-CAM Visualization for both models
resnet_model.eval()
vit_model.eval()

# Select one sample from each class (KL Grade 0-4)
fig, axes = plt.subplots(5, 4, figsize=(16, 20))
fig.suptitle('Grad-CAM Visualization: ResNet50 vs ViT', fontsize=16, fontweight='bold')

for class_idx in range(5):
    class_dir = os.path.join(TEST_DIR, str(class_idx))
    img_name = os.listdir(class_dir)[0]
    img_path = os.path.join(class_dir, img_name)

    # Original image processing
    original_img = Image.open(img_path).convert('RGB')
    original_img_resized = original_img.resize((IMG_SIZE, IMG_SIZE))

    # Preparation
    input_tensor, img_array = prepare_image_for_gradcam(img_path)
    input_tensor = input_tensor.to(device)

    # ResNet Grad-CAM
    target_layer_resnet = resnet_model.layer4[-1]
    cam_resnet = apply_gradcam(resnet_model, input_tensor, target_layer_resnet)

    # ViT Grad-CAM
    target_layer_vit = vit_model.encoder.layers[-1].ln_1
    cam_vit = apply_gradcam(vit_model, input_tensor, target_layer_vit, 
                            reshape_transform=vit_reshape_transform)

    # Visualization
    # Column 0: Original
    axes[class_idx, 0].imshow(original_img_resized)
    axes[class_idx, 0].set_title(f'Original (KL-{class_idx})')
    axes[class_idx, 0].axis('off')

    # Column 1: ResNet Grad-CAM overlay
    img_resized_array = np.array(original_img_resized) / 255.0
    cam_image_resnet = show_cam_on_image(img_resized_array, cam_resnet, use_rgb=True)
    axes[class_idx, 1].imshow(cam_image_resnet)
    axes[class_idx, 1].set_title('ResNet50 Grad-CAM')
    axes[class_idx, 1].axis('off')

    # Column 2: ViT Grad-CAM overlay
    cam_image_vit = show_cam_on_image(img_resized_array, cam_vit, use_rgb=True)
    axes[class_idx, 2].imshow(cam_image_vit)
    axes[class_idx, 2].set_title('ViT Grad-CAM')
    axes[class_idx, 2].axis('off')

    # Column 3: Raw Heatmap (ResNet)
    axes[class_idx, 3].imshow(cam_resnet, cmap='jet')
    axes[class_idx, 3].set_title('Raw Heatmap')
    axes[class_idx, 3].axis('off')

plt.tight_layout()
plt.savefig('gradcam_visualization.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Grad-CAM visualization saved successfully")

In [ ]:
from IPython.display import FileLink

# Це створить посилання, на яке можна натиснути
FileLink(r'vit_phase1_best.pth')

In [2]:
!kaggle kernels output kubyllka/knee-osteoarthritis -p /kaggle/working/

__notebook_source__.ipynb: Skipping, found more recently modified local copy (use --force to force download)
Output file downloaded to /kaggle/working/class_distribution.png
Output file downloaded to /kaggle/working/confusion_matrices.png
Output file downloaded to /kaggle/working/f1_loss_comparison.png
Output file downloaded to /kaggle/working/gradcam_visualization.png
Output file downloaded to /kaggle/working/resnet50_phase1_best.pth
Output file downloaded to /kaggle/working/resnet50_phase2_best.pth
Output file downloaded to /kaggle/working/runs/resnet50_phase1/events.out.tfevents.1776785667.9e9a20a96466.55.0
Output file downloaded to /kaggle/working/runs/resnet50_phase2/events.out.tfevents.1776786593.9e9a20a96466.55.1
Output file downloaded to /kaggle/working/runs/vit_phase1/events.out.tfevents.1776787767.9e9a20a96466.55.2
Output file downloaded to /kaggle/working/runs/vit_phase2/events.out.tfevents.1776788966.9e9a20a96466.55.3
Output file downloaded to /kaggle/working/runs/vit_phase

In [4]:
%load_ext tensorboard
%tensorboard --logdir runs

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6006 (pid 104), started 0:00:17 ago. (Use '!kill 104' to kill it.)

<IPython.core.display.Javascript object>

In [6]:
%tensorboard --logdir runs --port 6007

Reusing TensorBoard on port 6007 (pid 145), started 0:01:29 ago. (Use '!kill 145' to kill it.)

<IPython.core.display.Javascript object>